In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"]="1,2"

In [2]:
import pandas as pd
from datasets import Dataset

# Load your CSV
df = pd.read_csv('../data/tool_dataset/beavertails/data/330k/test/sampled/ratio_0.0.csv')

# Format as instruction-response pairs
# Option 1: Simple concatenation
# df['text'] = df.apply(lambda row: f"### Human: {row['prompt']}\n### Assistant: {row['response']}", axis=1)

# Option 2: Using Llama 2 chat format (recommended)
df['text'] = df.apply(lambda row: f"<s>[INST] {row['prompt']} [/INST] {row['response']} </s>", axis=1)

# Convert to HuggingFace Dataset
dataset = Dataset.from_pandas(df[['text']])

/home/michael920403/repos/fine-tuning-attack/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTConfig, SFTTrainer
import torch

max_length = 256

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("../models/Llama-2-7b-chat-hf")
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Optional: 4-bit quantization config (for memory efficiency)
# # Remove this section if you want to use full precision
# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_compute_dtype=torch.float16,
#     bnb_4bit_use_double_quant=True,
# )

# Load model
model = AutoModelForCausalLM.from_pretrained(
    "../models/Llama-2-7b-chat-hf",
    # quantization_config=bnb_config,  # Remove if not using 4-bit
    device_map="auto",
    torch_dtype=torch.float16,
)

# Prepare model for training (required for quantized models)
# model = prepare_model_for_kbit_training(model)


# Configure LoRA
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj"],  # Added o_proj
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

# Add LoRA adapters
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()  # Shows how many params are trainable

# Training config
training_args = SFTConfig(
    output_dir="../ft_models",
    num_train_epochs=4,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    # max_seq_length=512,
    dataset_text_field="text",
    fp16=True,  # Use fp16 if not using bf16
    # bf16=True,  # Use bf16 if your GPU supports it (A100, 3090, 4090, etc)
    logging_steps=10,
    save_strategy="epoch",
    optim="paged_adamw_8bit",  # Memory efficient optimizer
    gradient_checkpointing=True,
)

# Initialize trainer
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
)

# Train
trainer.train()

# Save the model
trainer.save_model("../ft_models/final")

Skipping import of cpp extensions due to incompatible torch version 2.8.0+cu128 for torchao version 0.14.0         Please see GitHub issue #2919 for more info
`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 2/2 [00:04<00:00,  2.33s/it]


trainable params: 6,291,456 || all params: 6,744,707,072 || trainable%: 0.0933


Truncating train dataset: 100%|██████████| 3000/3000 [00:00<00:00, 538283.37 examples/s]
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': None}.


Step,Training Loss
10,2.919500
20,2.813400
30,2.620300
40,2.431900
50,2.221100
60,2.062000
70,1.971700
80,1.922900
90,1.824300
100,1.732400
